# FinCrime-Capstone: Exploratory Data Analysis & Model Training

This notebook demonstrates the exploratory data analysis (EDA) and the training of our XGBoost model for detecting financial crime typologies. It pairs with our determinisitic Rule Engine to provide a robust Champion/Challenger detection layer.

## 1. Loading the Analytical Base Table

In [ ]:
import polars as pl
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

df = pl.read_parquet('../data/synthetic/analytical_base_table.parquet').to_pandas()
df.head()

## 2. Preprocessing & Feature Encoding

In [ ]:
feature_cols = ['amount', 'is_round_amount', 'txn_count_1d', 'total_amount_1d', 'avg_amount_1d', 'velocity_in_out_ratio']
cat_cols = ['risk_rating', 'jurisdiction']

for col in cat_cols:
    df[col] = df[col].astype(str)
    freq_encoding = df[col].value_counts(normalize=True)
    df[col + '_freq'] = df[col].map(freq_encoding)

final_features = feature_cols + [c + '_freq' for c in cat_cols]
X = df[final_features]
y = df['is_suspicious']

## 3. Training the XGBoost Model

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
model = xgb.XGBClassifier(n_estimators=100, max_depth=4, scale_pos_weight=scale_pos_weight, random_state=42)
model.fit(X_train, y_train)


## 4. SHAP Explainability
Financial crime models must be explainable. We use SHAP values to explain every decision to investigators.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, feature_names=final_features)